---
title: "Eventually Consistent Key-Value Store"
version: 2
abstract: >
    FIXME
syllabus:
-   Explain what "availability" and "partition tolerance" mean and why a system must choose between them and strong consistency during a network partition.
-   Describe how vector clocks track causal history and explain why taking the element-wise maximum is the correct merge operation.
-   Trace how a write's causal context is propagated so that a replica can detect out-of-order updates.
-   Identify the version explosion problem and explain when it becomes a practical concern.
---

**In Development**

The [CAP theorem][cap-theorem] says that when network partitions occur,
systems must choose between consistency (all nodes see the same data)
and availability (the system continues to respond).
Traditional databases choose consistency:
they stop accepting writes during a partition to avoid conflicts.
But for many applications,
availability is more valuable than immediate consistency.

[DynamoDB][dynamodb],
[Apache Cassandra][apache-cassandra],
and [Riak][riak] therefore take a different approach:
they remain available during partitions and accept that replicas might temporarily disagree.
They use techniques like vector clocks to track causality
and quorum protocols to balance consistency and availability.

This chapter builds a simplified eventually consistent key-value store to demonstrate these concepts.
It shows how systems handle concurrent writes,
detect conflicts using vector clocks,
and use read repair to synchronize replicas.

## The CAP Theorem in Practice {: #eventual-cap}

Imagine a key-value store with three replicas of each key:

1.  Client A writes "X = 1" to replicas 1 and 2.
1.  A network partition separates replica 3.
1.  Client B writes "X = 2" to replicas 2 and 3.
1.  The partition heals.

What is the value of X?
Replica 1 has "X = 1", replica 2 saw both writes, and replica 3 has "X = 2",
so there is a conflict.
Traditional databases would have prevented this by requiring all replicas to agree before accepting a write,
But that makes the system unavailable during the partition.

Our system accepts both writes and uses version vectors to track that X has two concurrent versions.
When a client reads X,
we can either return both versions and let the application decide how to resolve the conflict,
or use a simple rule like "last write wins" based on timestamps.

## Vector Clocks for Causality {: #eventual-clock}

Vector clocks let us determine if two events are causally related or concurrent.
Each replica maintains a counter for every replica in the system:

[%inc vector_clock.py mark=vectorclock %]

In our implementation,
the vector clock is a dictionary mapping replica IDs to integers.
When a replica performs an operation, it increments its own counter.
When a replica receives a message with a vector clock,
it takes the element-wise maximum of each component with its local clock.
"Element-wise maximum" means: for each replica ID in either clock,
the merged result uses the larger of the two values.
For example, merging `{"R1": 3, "R2": 1}` with `{"R1": 2, "R2": 4, "R3": 1}`
produces `{"R1": 3, "R2": 4, "R3": 1}`.

The key insight is that if clock A happens-before clock B,
then every entry in A's vector is ≤ the corresponding entry in B's vector,
with at least one entry strictly less.
If neither vector dominates the other element-wise,
the events are concurrent.

## Versioned Values {: #eventual-version}

Each value is stored with its vector clock:

[%inc versioned_value.py mark=versionedvalue %]

When storing multiple versions of a key,
we keep only the ones that are concurrent (neither happens-before the other).
If a new version happens-after an existing version,
we can discard the old one.

## Storage Node {: #eventual-storage}

Storage nodes can exchange four kinds of messages:

[%inc messages.py mark=messages %]

Each storage node maintains a replica of a subset of keys.
The node's constructor sets up its data store and creates the queue it uses to receive requests:

[%inc storage_node.py mark=storage_init %]

The `run` method dispatches reads and writes to dedicated handlers.
`_handle_read` returns all stored versions of a key:

[%inc storage_node.py mark=storage_read %]

The write handler is where causality is maintained.
It increments the local clock,
merges the client's causal context,
and discards any stored versions superseded by the new value.

"Merging the client's causal context" means the storage node takes the element-wise maximum
of its local vector clock and the clock the client sent.
The client's clock represents "what the client has already seen"—
it was returned to the client after a previous read.
By merging it in, the storage node ensures the new write is recorded as happening
*after* anything the client observed.
Without this step, a write that the client intended to follow a previous read
might appear concurrent with it, creating a spurious conflict.

[%inc storage_node.py mark=storage_write %]

When writing a new version, we:

1.  increment our local clock,
1.  merge the client's context (if provided) to preserve causality,
1.  remove any existing versions that are superseded by the new version, and
1.  keep concurrent versions, creating a conflict.

## Coordinator with Quorum Protocol {: #eventual-coord}

The coordinator manages replication across storage nodes.
It implements quorum reads and writes:
for N replicas,
we wait for R replicas to respond to a read and W replicas to respond to a write.
R + W > N guarantees that we see the latest write.

The constructor takes the list of storage nodes and quorum parameters,
and `_get_replicas` uses consistent hashing to choose which nodes store a given key:

[%inc coordinator.py mark=coord_init %]

A read sends requests to R replicas in parallel and merges their responses:

[%inc coordinator.py mark=coord_read %]

A write sends the new value to W replicas,
each of which updates its local clock.
The coordinator merges those clocks to return a causal context to the client:

[%inc coordinator.py mark=coord_write %]

`_merge_versions` resolves the list of all versions returned by replicas,
discarding any version whose clock is dominated by another:

[%inc coordinator.py mark=coord_merge %]

The quorum protocol is the heart of tunable consistency.
With N=3, R=2, W=2:

-   Writes succeed after 2 nodes acknowledge (available even if 1 node is down).
-   Reads query 2 nodes (at least one will have the latest write).
-   R + W = 4 > N = 3, ensuring reads see the latest writes.

## Client Implementation {: #eventual-client}

Clients read values, resolve conflicts, and write back.
The client stores a causal context per key and works through a list of operations:

[%inc kv_client.py mark=kv_init %]

When reading, the client checks for conflicts.
If a single version is returned there is no conflict.
If multiple concurrent versions are returned,
the client resolves them using last-write-wins and merges all their clocks:

[%inc kv_client.py mark=kv_read %]

When writing,
the client passes its current causal context so that
the coordinator can detect whether the write is concurrent with or follows previous writes:

[%inc kv_client.py mark=kv_write %]

The client maintains a context (vector clock) for each key.
When writing, it passes this context to preserve causality.
When reading multiple versions (a conflict),
it resolves using last-write-wins but merges all clocks to capture the complete causality.

## Running a Simulation {: #eventual-sim}

Let's create a scenario showing concurrent writes and conflict resolution:

[%inc ex_basic.py mark=basicexample %]
[%inc ex_basic.out %]

Now let's create a more interesting scenario with concurrent conflicting writes:

[%inc ex_conflict.py mark=conflictexample %]
[%inc ex_conflict.out %]

## Handling Network Partitions {: #eventual-partition}

Let's simulate a network partition to see how the system maintains availability.
`PartitionedCoordinator` extends the base coordinator with the ability to mark nodes as unreachable:

[%inc partitioned_coordinator.py mark=partition_init %]

Reads skip any partitioned nodes.
If fewer than R nodes are available the read fails;
otherwise it proceeds with only the reachable replicas:

[%inc partitioned_coordinator.py mark=partition_read %]

Writes follow the same pattern, requiring W reachable replicas before proceeding:

[%inc partitioned_coordinator.py mark=partition_write %]

Let's try it out:

[%inc ex_partition.py mark=partitionexample %]

## Read Repair {: #eventual-repair}

A real system needs mechanisms to ensure all replicas eventually converge.
Read repair happens during reads:
if we detect replicas are out of sync,
we push the latest version to lagging replicas.
`CoordinatorWithReadRepair` overrides `read` to query all replicas rather than just a quorum
so that it can identify which ones are behind:

[%inc coordinator_with_read_repair.py mark=repair_read %]

After collecting all responses,
`_perform_read_repair` compares each replica's set of versions against the merged result
and writes missing versions back to any replica that is out of date:

[%inc coordinator_with_read_repair.py mark=repair_perform %]

Read repair ensures that whenever we read a key,
we fix any inconsistencies we discover.
Over time, this brings all replicas into sync.

## Real-World Considerations {: #eventual-real}

Our implementation demonstrates core concepts, but production systems need additional features:

1.  Hinted handoff:
    when a node is temporarily down, writes intended for it are stored on another node with a hint.
    When the node recovers, the hints are replayed.

1.  Gossip protocol:
    nodes exchange information about cluster membership and failure detection through epidemic-style gossip.

1.  Compaction:
    Nodes merge version history periodically to avoid unbounded growth.
    This matters more than it might seem: if many concurrent writes reach the same key
    before any of them has been "dominated" by a later write,
    the number of concurrent versions grows.
    A system under heavy concurrent write load can accumulate hundreds of conflicting versions
    for a single key, making every subsequent read expensive.
    Real systems cap the number of concurrent versions and force conflict resolution
    when the cap is reached.

## Partition Example Output {: #eventual-partition-out}

Let's try the partition simulation:

[%inc ex_partition.py mark=partitionexample %]
[%inc ex_partition.out %]

The output shows that reads succeed while nodes are reachable (up to the quorum),
fail when too few nodes are available,
and succeed again after the partition heals.
Notice that the system does not lose data during the partition:
writes that succeeded on the available replicas
are picked up by the previously-partitioned replica during the next read repair cycle.

<section class="exercises" markdown="1">

## Exercises {: #eventual-exercises}

1.  The quorum parameters in the basic simulation are N=3, R=2, W=2.
    Change W to 1 and R to 3 and run the conflict scenario.
    What happens?
    Now try W=3 and R=1.
    For each configuration, describe a scenario where a client could read a stale value.

2.  Trace the vector clocks through this scenario manually (no code needed):
    - Three replicas: R1, R2, R3. All start with clock `{}`.
    - Client A writes "x=1" to R1. R1's clock becomes `{"R1": 1}`.
    - Client B writes "x=2" to R3. R3's clock becomes `{"R3": 1}`.
    - R1 and R3 merge. What is the resulting set of versions? Are they concurrent or is one dominated?
    - Client C then writes "x=3" to R2, passing the merged clock from the previous step.
      What is R2's clock after this write? Is x=1 or x=2 still in the version set?

3.  The write handler merges the client's causal context before storing the new version.
    Find the storage node's write handler in `storage_node.py`.
    Add a `print` statement that shows the client's clock, the node's clock before merging,
    and the node's clock after merging.
    Run the conflict scenario and explain the output.

4.  Read repair queries all replicas on every read.
    This is expensive—in a real system with thousands of keys and many replicas,
    you would not want to do it every time.
    Propose two strategies for reducing how often read repair runs.
    What consistency properties does each strategy sacrifice?

5.  The simulation uses last-write-wins (by timestamp) to resolve conflicts.
    Describe a scenario where this produces the wrong result from the application's perspective.
    What alternative conflict resolution strategy would work better for that scenario?

</section>